In [1]:
!ls /content

measuring_token_ids.npy				    sample_data
output_vocabulary_file_from_trained_tokenizer.json  train_token_ids.npy


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu130
True


In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [4]:
train_ids = np.load("/content/train_token_ids.npy")
measuring_ids = np.load("/content/measuring_token_ids.npy")

train_ids = torch.tensor(train_ids.astype(np.int64))
measuring_ids = torch.tensor(measuring_ids.astype(np.int64))

print(len(train_ids), len(measuring_ids))

128249 14064


In [5]:
context_length = 64
stride = 64
batch_size = 16

In [6]:
class TextChunks(Dataset):
    def __init__(self, ids, context_length, stride):
        self.inputs = []
        self.targets = []
        for start in range(0, len(ids) - context_length, stride):
            self.inputs.append(ids[start : start + context_length])
            self.targets.append(ids[start + 1 : start + context_length + 1])

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]

In [7]:
train_data = TextChunks(train_ids, context_length, stride)
measuring_data = TextChunks(measuring_ids, context_length, stride)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)
measuring_loader = DataLoader(measuring_data, batch_size=batch_size, shuffle=False, drop_last=False)

print(len(train_data), len(measuring_data))
print(len(train_loader), len(measuring_loader))

2003 219
125 14


In [8]:
x, y = next(iter(train_loader))
print(x.shape, y.shape)
print((x[:, 1:] == y[:, :-1]).all())

torch.Size([16, 64]) torch.Size([16, 64])
tensor(True)


In [9]:
!pip install -q tokenizers
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("/content/output_vocabulary_file_from_trained_tokenizer.json")
print(tokenizer.decode(x[0].tolist(), skip_special_tokens=False))

enjoy it . [EOS] Oh yes , it ' s a well no it ' s quite it ' s quite difficult . [EOS] I prefer reading fiction books . [EOS] When I try to stay focused , many things can distract me . It ' s like some noise , some interesting things , or just a phone call , you know .


In [10]:
config = {
    "vocab_size": 6134,
    "context_length": 64,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.2,
}

In [11]:
token_embedding = torch.nn.Embedding(config["vocab_size"], config["emb_dim"])
position_embedding = torch.nn.Embedding(config["context_length"], config["emb_dim"])

token_vectors = token_embedding(x)
positions = torch.arange(config["context_length"])
position_vectors = position_embedding(positions)
combined = token_vectors + position_vectors

print(token_vectors.shape, position_vectors.shape, combined.shape)

torch.Size([16, 64, 128]) torch.Size([64, 128]) torch.Size([16, 64, 128])


In [12]:
class CausalSelfAttention(torch.nn.Module):
    def __init__(self, emb_dim, n_heads, context_length, drop_rate):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads
        self.query = torch.nn.Linear(emb_dim, emb_dim)
        self.key = torch.nn.Linear(emb_dim, emb_dim)
        self.value = torch.nn.Linear(emb_dim, emb_dim)
        self.out = torch.nn.Linear(emb_dim, emb_dim)
        self.dropout = torch.nn.Dropout(drop_rate)
        mask = torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        self.register_buffer("mask", mask)

    def forward(self, x):
        batch, length, emb_dim = x.shape
        q = self.query(x).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)

        scores = q @ k.transpose(2, 3) / (self.head_dim ** 0.5)
        scores = scores.masked_fill(self.mask[:length, :length], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        result = (weights @ v).transpose(1, 2).reshape(batch, length, emb_dim)
        return self.out(result)

In [13]:
attention = CausalSelfAttention(config["emb_dim"], config["n_heads"], config["context_length"], config["drop_rate"])
attention_output = attention(combined)
print(attention_output.shape)

torch.Size([16, 64, 128])


In [14]:
attention.eval()
test_input = combined.detach().clone()
out_before = attention(test_input)
test_input[:, -1, :] = test_input[:, -1, :] + 5
out_after = attention(test_input)
print(torch.allclose(out_before[:, :-1, :], out_after[:, :-1, :]))

True


In [15]:
class FeedForward(torch.nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(emb_dim, 4 * emb_dim),
            torch.nn.GELU(),
            torch.nn.Linear(4 * emb_dim, emb_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(config["emb_dim"])
        self.attention = CausalSelfAttention(config["emb_dim"], config["n_heads"], config["context_length"], config["drop_rate"])
        self.norm2 = torch.nn.LayerNorm(config["emb_dim"])
        self.feed_forward = FeedForward(config["emb_dim"])
        self.dropout = torch.nn.Dropout(config["drop_rate"])

    def forward(self, x):
        x = x + self.dropout(self.attention(self.norm1(x)))
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x

In [16]:
class SmallGPT(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = torch.nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.position_embedding = torch.nn.Embedding(config["context_length"], config["emb_dim"])
        self.dropout = torch.nn.Dropout(config["drop_rate"])
        self.blocks = torch.nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])
        self.final_norm = torch.nn.LayerNorm(config["emb_dim"])
        self.output_layer = torch.nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, ids):
        batch, length = ids.shape
        positions = torch.arange(length, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)
        x = self.dropout(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.output_layer(x)

In [17]:
model = SmallGPT(config)
logits = model(x)
print(logits.shape)

total_parameters = sum(p.numel() for p in model.parameters())
print(total_parameters)

loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())
print(loss.item())

torch.Size([16, 64, 6134])
2371840
8.899508476257324


In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

torch.manual_seed(123)
model = SmallGPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

Training on: cuda


In [19]:
def batch_loss(x, y):
    x, y = x.to(device), y.to(device)
    logits = model(x)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())


def average_loss(loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in loader:
            total += batch_loss(x, y).item()
    model.train()
    return total / len(loader)

In [20]:
import time

max_epochs = 30
best_measuring_loss = float("inf")
history = []

for epoch in range(1, max_epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = batch_loss(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    measuring_loss = average_loss(measuring_loader)
    history.append((epoch, train_loss, measuring_loss))

    if measuring_loss < best_measuring_loss:
        best_measuring_loss = measuring_loss
        torch.save(model.state_dict(), "/content/best_model.pt")
        note = "  <- saved (best so far)"
    else:
        note = ""

    print(f"Epoch {epoch}: train loss {train_loss:.3f}, measuring loss {measuring_loss:.3f}, {time.time() - start_time:.1f}s{note}")

Epoch 1: train loss 6.059, measuring loss 5.381, 2.9s  <- saved (best so far)
Epoch 2: train loss 5.130, measuring loss 5.044, 1.8s  <- saved (best so far)
Epoch 3: train loss 4.828, measuring loss 4.918, 1.8s  <- saved (best so far)
Epoch 4: train loss 4.624, measuring loss 4.844, 1.9s  <- saved (best so far)
Epoch 5: train loss 4.461, measuring loss 4.803, 2.4s  <- saved (best so far)
Epoch 6: train loss 4.317, measuring loss 4.772, 2.2s  <- saved (best so far)
Epoch 7: train loss 4.185, measuring loss 4.757, 1.8s  <- saved (best so far)
Epoch 8: train loss 4.065, measuring loss 4.749, 1.9s  <- saved (best so far)
Epoch 9: train loss 3.947, measuring loss 4.745, 1.8s  <- saved (best so far)
Epoch 10: train loss 3.832, measuring loss 4.769, 1.8s
Epoch 11: train loss 3.731, measuring loss 4.788, 2.1s
Epoch 12: train loss 3.629, measuring loss 4.812, 2.4s
Epoch 13: train loss 3.537, measuring loss 4.860, 1.8s
Epoch 14: train loss 3.454, measuring loss 4.879, 2.0s
Epoch 15: train loss 3.

In [21]:
from google.colab import files
files.download("/content/best_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
model.eval()

def generate(start_text, max_new_tokens=40, temperature=0.8, top_k=20):
    ids = tokenizer.encode(start_text).ids
    ids = torch.tensor([ids], device=device)
    for _ in range(max_new_tokens):
        context = ids[:, -config["context_length"]:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        logits = logits / temperature
        top_values, _ = torch.topk(logits, top_k)
        logits[logits < top_values[:, -1:]] = float("-inf")
        probabilities = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        if next_id.item() == 2:
            break
        ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids[0].tolist())

In [23]:
torch.manual_seed(1)
for start in ["I think", "Yes, of course", "Yesterday I", "I don't"]:
    print(start, "->", generate(start))
    print()

I think -> I think I think so .

Yes, of course -> Yes , of course .

Yesterday I -> Y esterday I don ' t like it ' s necessary . It ' s really really good life .

I don't -> I don ' t remember it and I would like to keep it . So maybe I think it is so I think it ' s not be very like the teacher because the library because I think it ' s not a good



In [24]:
import json
import random
from collections import Counter

with open("/content/cleaned_transcripts.json", "r", encoding="utf-8") as f:
    records = json.load(f)

print(len(records))
print(records[0].keys())

indices = list(range(len(records)))
random.seed(42)
random.shuffle(indices)
measuring_indices = set(indices[:int(len(records) * 0.10)])

train_labeled = []
measuring_labeled = []
for i, record in enumerate(records):
    label = record["emotion label"]
    if label is None:
        continue
    item = (label, record["corrected whisper_transcript"])
    if i in measuring_indices:
        measuring_labeled.append(item)
    else:
        train_labeled.append(item)

print(len(train_labeled), len(measuring_labeled))
print(Counter(label for label, text in train_labeled))
print(Counter(label for label, text in measuring_labeled))

3081
dict_keys(['Unit', 'emotion label', 'corrected whisper_transcript'])
228 28
Counter({'Neutral': 146, 'Pauses': 47, 'Negative': 35})
Counter({'Neutral': 22, 'Pauses': 5, 'Negative': 1})


In [25]:
tag_for_label = {"Neutral": "[NEUTRAL]", "Negative": "[NEGATIVE]", "Pauses": "[PAUSES]"}

def make_examples(labeled_items):
    inputs, targets = [], []
    for label, text in labeled_items:
        line = tag_for_label[label] + " " + text + " [EOS]"
        ids = tokenizer.encode(line).ids[: config["context_length"] + 1]
        ids = ids + [1] * (config["context_length"] + 1 - len(ids))
        ids = torch.tensor(ids)
        inputs.append(ids[:-1])
        targets.append(ids[1:])
    return torch.stack(inputs), torch.stack(targets)

ft_train_x, ft_train_y = make_examples(train_labeled)
ft_measuring_x, ft_measuring_y = make_examples(measuring_labeled)

print(ft_train_x.shape, ft_measuring_x.shape)
print(tokenizer.decode(ft_train_x[0].tolist(), skip_special_tokens=False))

torch.Size([228, 64]) torch.Size([28, 64])
[NEUTRAL] No body . [EOS] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [26]:
from torch.utils.data import TensorDataset

ft_train_loader = DataLoader(TensorDataset(ft_train_x, ft_train_y), batch_size=16, shuffle=True)
ft_measuring_loader = DataLoader(TensorDataset(ft_measuring_x, ft_measuring_y), batch_size=16, shuffle=False)

print(len(ft_train_loader), len(ft_measuring_loader))

15 2


In [27]:
def ft_batch_loss(x, y):
    x, y = x.to(device), y.to(device)
    logits = model(x)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten(), ignore_index=1)


def ft_average_loss(loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for x, y in loader:
            total += ft_batch_loss(x, y).item()
    model.train()
    return total / len(loader)

In [28]:
model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
ft_optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

print("Before fine-tuning: measuring loss", round(ft_average_loss(ft_measuring_loader), 3))

best_ft_loss = float("inf")
for epoch in range(1, 16):
    model.train()
    running = 0.0
    for x, y in ft_train_loader:
        ft_optimizer.zero_grad()
        loss = ft_batch_loss(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        ft_optimizer.step()
        running += loss.item()

    train_loss = running / len(ft_train_loader)
    measuring_loss = ft_average_loss(ft_measuring_loader)
    note = ""
    if measuring_loss < best_ft_loss:
        best_ft_loss = measuring_loss
        torch.save(model.state_dict(), "/content/best_finetuned_model.pt")
        note = "  <- saved (best so far)"
    print(f"Epoch {epoch}: train loss {train_loss:.3f}, measuring loss {measuring_loss:.3f}{note}")

Before fine-tuning: measuring loss 4.61
Epoch 1: train loss 3.987, measuring loss 4.442  <- saved (best so far)
Epoch 2: train loss 3.797, measuring loss 4.431  <- saved (best so far)
Epoch 3: train loss 3.661, measuring loss 4.449
Epoch 4: train loss 3.566, measuring loss 4.460
Epoch 5: train loss 3.499, measuring loss 4.477
Epoch 6: train loss 3.380, measuring loss 4.494
Epoch 7: train loss 3.347, measuring loss 4.514
Epoch 8: train loss 3.235, measuring loss 4.532
Epoch 9: train loss 3.175, measuring loss 4.546
Epoch 10: train loss 3.097, measuring loss 4.566
Epoch 11: train loss 3.043, measuring loss 4.587
Epoch 12: train loss 3.005, measuring loss 4.608
Epoch 13: train loss 2.942, measuring loss 4.629
Epoch 14: train loss 2.866, measuring loss 4.653
Epoch 15: train loss 2.821, measuring loss 4.670


In [29]:
model.load_state_dict(torch.load("/content/best_finetuned_model.pt", map_location=device))

def loss_with_labels(items):
    ex_x, ex_y = make_examples(items)
    loader = DataLoader(TensorDataset(ex_x, ex_y), batch_size=16)
    return ft_average_loss(loader)

print("Correct tags:", round(loss_with_labels(measuring_labeled), 3))
for tag_label in ["Neutral", "Negative", "Pauses"]:
    all_same = [(tag_label, text) for label, text in measuring_labeled]
    print("Every transcript tagged", tag_label + ":", round(loss_with_labels(all_same), 3))

Correct tags: 4.431
Every transcript tagged Neutral: 4.431
Every transcript tagged Negative: 4.432
Every transcript tagged Pauses: 4.43


In [30]:
model.eval()
torch.manual_seed(2)
for tag in ["[NEUTRAL]", "[NEGATIVE]", "[PAUSES]"]:
    print(tag)
    for start in ["I think", "I don't", "Yes"]:
        print("  ", start, "->", generate(tag + " " + start))
    print()

[NEUTRAL]
   I think -> I think it ' s it ' s easy to make me to be very difficult for us .
   I don't -> I don ' t think it ' s very important for me to study with others . And in the morning . And I have to do that I don ' t see the teacher and then I don ' t want to go
   Yes -> Yes , of course , and I don ' t have been the same thing .

[NEGATIVE]
   I think -> I think about my parents and my friends and my friends . I have to go to school and my hometown .
   I don't -> I don ' t want to the time when I ' t like .
   Yes -> Yes , I ' d like to talk to work . I don ' t need to read books . And I don ' t think it ' s important for it ' s really important for me .

[PAUSES]
   I think -> I think it can bring me in my time .
   I don't -> I don ' t didn ' t have no about the day .
   Yes -> Yes , I know , I think it ' s very much a great place to understand what kind of course .



In [31]:
from google.colab import files
files.download("/content/best_finetuned_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
non_neutral_train = [(label, text) for label, text in train_labeled if label != "Neutral"]
non_neutral_measuring = [(label, text) for label, text in measuring_labeled if label != "Neutral"]

def tag_gap(items):
    correct = loss_with_labels(items)
    as_neutral = loss_with_labels([("Neutral", text) for label, text in items])
    return as_neutral - correct

print(len(non_neutral_train), len(non_neutral_measuring))

82 6


In [33]:
model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
ft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

print("Start: measuring loss", round(ft_average_loss(ft_measuring_loader), 3),
      "| train tag gap", round(tag_gap(non_neutral_train), 4),
      "| measuring tag gap", round(tag_gap(non_neutral_measuring), 4))

for epoch in range(1, 16):
    model.train()
    running = 0.0
    for x, y in ft_train_loader:
        ft_optimizer.zero_grad()
        loss = ft_batch_loss(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        ft_optimizer.step()
        running += loss.item()

    train_loss = running / len(ft_train_loader)
    measuring_loss = ft_average_loss(ft_measuring_loader)
    train_gap = tag_gap(non_neutral_train)
    measuring_gap = tag_gap(non_neutral_measuring)
    torch.save(model.state_dict(), f"/content/finetuned_epoch_{epoch}.pt")
    print(f"Epoch {epoch}: train loss {train_loss:.3f}, measuring loss {measuring_loss:.3f}, "
          f"train tag gap {train_gap:.4f}, measuring tag gap {measuring_gap:.4f}")

Start: measuring loss 4.61 | train tag gap -0.048 | measuring tag gap -0.0358
Epoch 1: train loss 4.006, measuring loss 4.474, train tag gap 0.0133, measuring tag gap -0.0033
Epoch 2: train loss 3.543, measuring loss 4.523, train tag gap 0.0228, measuring tag gap -0.0056
Epoch 3: train loss 3.296, measuring loss 4.587, train tag gap 0.0276, measuring tag gap -0.0023
Epoch 4: train loss 3.119, measuring loss 4.668, train tag gap 0.0372, measuring tag gap -0.0038
Epoch 5: train loss 2.885, measuring loss 4.716, train tag gap 0.0434, measuring tag gap -0.0033
Epoch 6: train loss 2.776, measuring loss 4.771, train tag gap 0.0628, measuring tag gap 0.0004
Epoch 7: train loss 2.612, measuring loss 4.845, train tag gap 0.0546, measuring tag gap -0.0035
Epoch 8: train loss 2.484, measuring loss 4.911, train tag gap 0.0926, measuring tag gap -0.0060
Epoch 9: train loss 2.331, measuring loss 4.976, train tag gap 0.0844, measuring tag gap -0.0055
Epoch 10: train loss 2.210, measuring loss 5.032, 

In [34]:
random.seed(7)
shuffled_non_neutral = non_neutral_train[:]
random.shuffle(shuffled_non_neutral)

held_out_items = shuffled_non_neutral[:20] + non_neutral_measuring

neutral_train = [(label, text) for label, text in train_labeled if label == "Neutral"]
new_ft_train = neutral_train + shuffled_non_neutral[20:]

new_x, new_y = make_examples(new_ft_train)
new_loader = DataLoader(TensorDataset(new_x, new_y), batch_size=16, shuffle=True)

print(len(new_ft_train), len(held_out_items))
print(Counter(label for label, text in held_out_items))

208 26
Counter({'Pauses': 18, 'Negative': 8})


In [35]:
def run_finetune(lr, epochs=15):
    model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    print(f"--- learning rate {lr} ---")
    print("Start: held-out tag gap", round(tag_gap(held_out_items), 4))
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for x, y in new_loader:
            optimizer.zero_grad()
            loss = ft_batch_loss(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
        print(f"Epoch {epoch}: train loss {running / len(new_loader):.3f}, "
              f"measuring loss {ft_average_loss(ft_measuring_loader):.3f}, "
              f"held-out tag gap {tag_gap(held_out_items):.4f}")

run_finetune(3e-4)
run_finetune(1e-3)

--- learning rate 0.0003 ---
Start: held-out tag gap -0.0495
Epoch 1: train loss 4.032, measuring loss 4.453, held-out tag gap 0.0085
Epoch 2: train loss 3.799, measuring loss 4.434, held-out tag gap -0.0064
Epoch 3: train loss 3.681, measuring loss 4.450, held-out tag gap -0.0018
Epoch 4: train loss 3.576, measuring loss 4.459, held-out tag gap -0.0012
Epoch 5: train loss 3.498, measuring loss 4.477, held-out tag gap -0.0014
Epoch 6: train loss 3.389, measuring loss 4.490, held-out tag gap 0.0000
Epoch 7: train loss 3.324, measuring loss 4.507, held-out tag gap -0.0008
Epoch 8: train loss 3.247, measuring loss 4.525, held-out tag gap -0.0003
Epoch 9: train loss 3.179, measuring loss 4.538, held-out tag gap 0.0012
Epoch 10: train loss 3.118, measuring loss 4.558, held-out tag gap 0.0009
Epoch 11: train loss 3.060, measuring loss 4.573, held-out tag gap 0.0018
Epoch 12: train loss 2.983, measuring loss 4.596, held-out tag gap 0.0014
Epoch 13: train loss 2.924, measuring loss 4.622, held

In [36]:
all_labeled = train_labeled + measuring_labeled
holdout_sizes = {"Pauses": 10, "Negative": 8, "Neutral": 20}

random.seed(11)
specialist_train = {}
specialist_heldout = {}
for label, size in holdout_sizes.items():
    items = [text for l, text in all_labeled if l == label]
    random.shuffle(items)
    specialist_heldout[label] = items[:size]
    specialist_train[label] = items[size:]
    print(label, "fine-tune on", len(items) - size, "| held out", size)

Pauses fine-tune on 42 | held out 10
Negative fine-tune on 28 | held out 8
Neutral fine-tune on 148 | held out 20


In [37]:
def make_plain_examples(texts):
    inputs, targets = [], []
    for text in texts:
        ids = tokenizer.encode(text + " [EOS]").ids[: config["context_length"] + 1]
        ids = ids + [1] * (config["context_length"] + 1 - len(ids))
        ids = torch.tensor(ids)
        inputs.append(ids[:-1])
        targets.append(ids[1:])
    return torch.stack(inputs), torch.stack(targets)

def plain_loader(texts, shuffle):
    px, py = make_plain_examples(texts)
    return DataLoader(TensorDataset(px, py), batch_size=16, shuffle=shuffle)

heldout_loaders = {label: plain_loader(texts, False) for label, texts in specialist_heldout.items()}

In [38]:
labels = ["Pauses", "Negative", "Neutral"]

def table_row(name):
    scores = [round(ft_average_loss(heldout_loaders[l]), 3) for l in labels]
    print(f"{name:<24}", scores)

def train_specialist(label, epochs=6, lr=3e-4):
    model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    loader = plain_loader(specialist_train[label], True)
    print(f"--- {label} copy ---")
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in loader:
            optimizer.zero_grad()
            loss = ft_batch_loss(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        table_row(f"after epoch {epoch}")
        torch.save(model.state_dict(), f"/content/specialist_{label}_epoch{epoch}.pt")

model.load_state_dict(torch.load("/content/best_model.pt", map_location=device))
print("columns are held-out loss for:", labels)
table_row("pretrained, no tuning")
for label in labels:
    train_specialist(label)

columns are held-out loss for: ['Pauses', 'Negative', 'Neutral']
pretrained, no tuning    [4.018, 3.783, 3.785]
--- Pauses copy ---
after epoch 1            [4.034, 3.787, 3.775]
after epoch 2            [4.05, 3.809, 3.783]
after epoch 3            [4.061, 3.834, 3.789]
after epoch 4            [4.066, 3.855, 3.796]
after epoch 5            [4.073, 3.871, 3.805]
after epoch 6            [4.082, 3.889, 3.815]
--- Negative copy ---
after epoch 1            [4.015, 3.689, 3.771]
after epoch 2            [4.024, 3.616, 3.775]
after epoch 3            [4.038, 3.567, 3.788]
after epoch 4            [4.056, 3.536, 3.807]
after epoch 5            [4.071, 3.513, 3.828]
after epoch 6            [4.083, 3.49, 3.845]
--- Neutral copy ---
after epoch 1            [4.021, 3.747, 3.769]
after epoch 2            [4.003, 3.768, 3.757]
after epoch 3            [4.004, 3.8, 3.764]
after epoch 4            [4.016, 3.818, 3.776]
after epoch 5            [4.025, 3.842, 3.775]
after epoch 6            [4.03

In [39]:
def samples_from(checkpoint, title):
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()
    torch.manual_seed(5)
    print(title)
    for start in ["I think", "I don't", "Yes"]:
        for _ in range(2):
            print("  ", start, "->", generate(start))
    print()

samples_from("/content/specialist_Neutral_epoch2.pt", "NEUTRAL copy (epoch 2)")
samples_from("/content/specialist_Negative_epoch6.pt", "NEGATIVE copy (epoch 6)")

NEUTRAL copy (epoch 2)
   I think -> I think it ' s important for me to do it ' s very important .
   I think -> I think it ' s more good for me and you don ' t want to see a nap . That ' s really good way to have to have the whole .
   I don't -> I don ' t think I ' t have no idea about this person .
   I don't -> I don ' t want to go to the library .
   Yes -> Yes , I ' m a good way to be busy .
   Yes -> Yes , I can ' t think I don ' t have many old school because I like to have .

NEGATIVE copy (epoch 6)
   I think -> I think it ' s important for me to understand it ' s very important and I don ' t have a lot of the answer . And you can ' t understand what you know the big ?
   I think -> I think it ' s so maybe you said ?
   I don't -> I don ' t know what ' t understand ?
   I don't -> I don ' t think it ' t have no , so it , but I don ' t understand .
   Yes -> Yes , I will be something about this .
   Yes -> Yes , I ' m not a child .



In [40]:
train_specialist("Negative", epochs=14)

--- Negative copy ---
after epoch 1            [4.017, 3.682, 3.772]
after epoch 2            [4.024, 3.607, 3.776]
after epoch 3            [4.037, 3.562, 3.785]
after epoch 4            [4.053, 3.532, 3.799]
after epoch 5            [4.065, 3.509, 3.813]
after epoch 6            [4.074, 3.488, 3.826]
after epoch 7            [4.08, 3.47, 3.838]
after epoch 8            [4.086, 3.452, 3.849]
after epoch 9            [4.091, 3.435, 3.861]
after epoch 10           [4.096, 3.424, 3.875]
after epoch 11           [4.101, 3.418, 3.889]
after epoch 12           [4.107, 3.417, 3.905]
after epoch 13           [4.115, 3.417, 3.921]
after epoch 14           [4.124, 3.418, 3.937]


In [41]:
def generate_ids(start_text, max_new_tokens=40, temperature=0.8, top_k=20):
    ids = torch.tensor([tokenizer.encode(start_text).ids], device=device)
    for _ in range(max_new_tokens):
        context = ids[:, -config["context_length"]:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        logits = logits / temperature
        top_values, _ = torch.topk(logits, top_k)
        logits[logits < top_values[:, -1:]] = float("-inf")
        probabilities = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        if next_id.item() == 2:
            break
        ids = torch.cat([ids, next_id], dim=1)
    return ids[0].tolist()

def twelve_grams(ids):
    return {tuple(ids[i:i + 12]) for i in range(len(ids) - 11)}

all_runs = set()
for record in records:
    all_runs |= twelve_grams(tokenizer.encode(record["corrected whisper_transcript"]).ids)

def copy_check(name, checkpoint, n_samples=50):
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()
    torch.manual_seed(9)
    starters = ["I think", "I don't", "Yes", "I"]
    copied = 0
    for i in range(n_samples):
        ids = generate_ids(starters[i % len(starters)])
        if twelve_grams(ids) & all_runs:
            copied += 1
    print(name, "-", copied, "of", n_samples, "samples contain a 12-token run copied from a real transcript")

copy_check("Neutral copy", "/content/specialist_Neutral_epoch2.pt")
copy_check("Negative copy", "/content/specialist_Negative_epoch10.pt")

Neutral copy - 0 of 50 samples contain a 12-token run copied from a real transcript
Negative copy - 0 of 50 samples contain a 12-token run copied from a real transcript


In [42]:
from google.colab import files
files.download("/content/specialist_Neutral_epoch2.pt")
files.download("/content/specialist_Negative_epoch10.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
from google.colab import files
files.download("/content/specialist_Negative_epoch10.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>